# 13g - Reconciliation Checks (C2 / C6 / C7 / C8 + OOF inventory) - v2

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.


Resolves the thesis_build contradictions empirically from on-disk data. Read-only; writes nothing.
Runs in WSL/conda/JupyterLab via global_setup (no hardcoded Windows paths).

- **C2** feature count: varies by (dataset family) x (sampling unit V2 buildings vs V3-V7 points) x (tier)
- **C6** match radius (point -> building): 10 m operative vs 15 m diagnostic-label discrepancy
- **C7** V3 point row count (raw vs filtered)
- **C8** y_true unit (building-level vs point-level positives)
- **Footprint**: Overture (primary) vs OSM (comparison) - reliability note
- **OOF inventory** across BOTH roots (results/ and data/outputs/) - coverage for N1

In [1]:
import sys, collections
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# locate global_setup cross-platform (WSL / Windows): walk up from cwd, then known roots
_known = [Path("/content/drive_f/masterthesis/notebooks"),
          Path("/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks"),
          Path(r"F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks")]
_nb = next((c for c in list(Path.cwd().parents) + _known if (c / "global_setup.py").exists()), None)
if _nb is None:
    raise RuntimeError("global_setup.py not found - run from inside the notebooks tree")
if str(_nb) not in sys.path:
    sys.path.insert(0, str(_nb))
import global_setup as gs

DRIVE_ROOT   = Path(gs.DRIVE_ROOT)
STACK_ROOT   = Path(gs.STACK_ROOT)
RESULTS_ROOT = Path(gs.RESULTS_ROOT)
DS = STACK_ROOT / "dataset"
TIERS = [0, 1, 2]
print("notebooks:", _nb)
print("DATASET :", DS)
print("RESULTS :", RESULTS_ROOT)

def cols(p):
    try: return [f.name for f in pq.ParquetFile(p).schema_arrow]
    except Exception: return []
def nrows(p):
    try: return pq.ParquetFile(p).metadata.num_rows
    except Exception: return None

/home/alpineobotics/miniconda3/envs/bda/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


BDA GLOBAL SETUP
Started: 2026-07-03 08:44:33
Python: 3.12.12

[1/7] Directory Structure
----------------------------------------------------------------------
  GDrive (G:):       /content/drive_f/masterthesis OK
  GDrive (F:):       /content/drive_f/masterthesis OK
  Local data (G:):   /content/masterthesis_local/data OK
  Data stack (F:):   /mnt/f/PROJECTS/masterthesis/data_stack OK

  TIER_SELECTION: [0, 1, 2]
  CITY_SELECTION: None (tier filter)
  REQUIRE_UNOSAT: False
  CITIES_TO_PROCESS: 21 cities

[2/7] Credentials
----------------------------------------------------------------------
  Copernicus: inf***
  OpenTopography: OK
  Earthdata: marcoheinzen

[3/7] Python Packages
----------------------------------------------------------------------


/content/drive_f/masterthesis/notebooks/global_setup.py:564: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources



  Already installed: 23
  Newly installed:   0
  Failed:            0

[4/7] Global Imports & Configuration
----------------------------------------------------------------------
  All imports loaded

[5/7] Processing Config & SNAP
----------------------------------------------------------------------
  GPT: Usage:
  Temporal baseline: 10-24 days
  Wavelength: 0.0555

[6/7] GPU Status
----------------------------------------------------------------------
  CUDA available: NVIDIA GeForce RTX 2070 SUPER
    CUDA version: 12.8

[7/7] Disk Space
----------------------------------------------------------------------
  GDrive (G:)     930.7/7452.0 GB (6521.3 GB free)
  GDrive (F:)     1405.7/3726.0 GB (2320.3 GB free)
  Local data      11563.4/14901.9 GB (3338.4 GB free)
  Data stack      1405.7/3726.0 GB (2320.3 GB free)
  WSL ext4        69.1/1006.9 GB (886.6 GB free)

GLOBAL SETUP COMPLETE
  Torch device: cuda
  Cities: 21, CITY=Avdiivka
  Functions: load_aoi(), load_aoi_gdf(), load_aoi_

## C2 - feature count by dataset family x sampling unit x tier

No single "feature count" exists: block_stats is wide-format, so column count scales with how many
scenes/observations a tier's cities have. It also differs between V2 (buildings) and V3-V7 (points) and between
families. The ML-relevant number is the **shared (intersection) feature set across tiers** (GroupKFold needs columns
present in every fold). "dietrich28" is a column subset, not a separate parquet.

In [2]:
ID_LABEL = {"unosat_ep","unosat_id","unosat_date","damage_label","damage","ems98_grade",
            "damage_binary","match_distance","match_method","in_aoi","city","building_id",
            "point_id","tier","battle_start","battle_stop","conflict_ongoing",
            "has_card","has_coh","has_ms","centroid_x","centroid_y","x_utm","y_utm",
            "lon","lat","row","col","point_source","ep","t_unosat","n_pixels",
            "height","roof_height","num_floors","area_m2"}

families = ["bda_block_stats","bda_composite_prepost_bands","bda_block_accum_card",
            "bda_block_accum_coh","bda_block_accum_ms","bda_fusion_composite_blockstats",
            "bda_dietrich","bda_prepost_single_card","bda_rolling_stats_card","bda_rolling_stats_coh"]

rows = []
for sub in ["V2","v3","v4","v5","v6","v7"]:
    root = DS / sub
    if not root.exists(): continue
    for fam in families:
        for t in TIERS:
            p = root / f"{fam}_t{t}.parquet"
            if p.exists():
                c = cols(p); feats = [x for x in c if x not in ID_LABEL]
                rows.append({"sub":sub,"family":fam,"tier":t,"n_cols":len(c),"n_features":len(feats)})
fc = pd.DataFrame(rows)
if len(fc):
    print("FEATURE COUNT by family x sampling-unit x tier:\n")
    print(fc.to_string(index=False))
else:
    print("no matching parquets found")

print("\nSHARED feature columns across tiers (what GroupKFold ML actually uses):")
for sub in (fc["sub"].unique() if len(fc) else []):
    for fam in fc[fc["sub"]==sub]["family"].unique():
        per_tier = []
        for t in TIERS:
            p = DS / sub / f"{fam}_t{t}.parquet"
            if p.exists(): per_tier.append(set(x for x in cols(p) if x not in ID_LABEL))
        if len(per_tier) >= 2:
            shared = set.intersection(*per_tier)
            print(f"  {sub:3s} {fam:34s} tiers={len(per_tier)} shared_features={len(shared)}")

FEATURE COUNT by family x sampling-unit x tier:

sub                          family  tier  n_cols  n_features
 V2                 bda_block_stats     0     745         743
 V2                 bda_block_stats     1     183         181
 V2                 bda_block_stats     2     897         895
 V2     bda_composite_prepost_bands     0     586         584
 V2     bda_composite_prepost_bands     1     192         190
 V2     bda_composite_prepost_bands     2     192         190
 V2            bda_block_accum_card     0      93          91
 V2             bda_block_accum_coh     0      29          27
 V2              bda_block_accum_ms     0    1275        1273
 V2 bda_fusion_composite_blockstats     0    1330        1328
 V2 bda_fusion_composite_blockstats     1     374         372
 V2 bda_fusion_composite_blockstats     2    1088        1086
 V2         bda_prepost_single_card     0      60          57
 V2         bda_prepost_single_card     1      18          15
 V2         bda_prepo

## C6 - match radius (point -> building), 10 m vs 15 m

NB01 (01a/01b) sets `MAX_DISTANCE_METERS = 10` for the operative link; a later **diagnostic** cell reassigns `= 15`
and plot axvlines hardcode "Current threshold (15m)". Provable from `match_distance`: if its max is <= 10 m, the link
threshold was 10 m and 15 m is only a stale diagnostic label.

In [3]:
parts = []
for t in TIERS:
    p = DS / "V2" / f"bda_buildings_t{t}.parquet"
    if p.exists():
        parts.append(pd.read_parquet(p, columns=["match_distance","match_method","damage_binary","city"]))
if parts:
    b = pd.concat(parts, ignore_index=True)
    md_ = b["match_distance"].dropna()
    print(f"match_distance present: {len(md_):,} of {len(b):,} rows")
    if len(md_):
        print(f"  max={md_.max():.3f} m | >10m={int((md_>10).sum())} | >15m={int((md_>15).sum())}")
    print("  match_method counts:")
    print(b["match_method"].value_counts(dropna=False).to_string())
    verdict = "10 m" if (len(md_) and md_.max() <= 10.0001) else f"UNCERTAIN (max={md_.max() if len(md_) else 'NA'})"
    print(f"\n  OPERATIVE RADIUS = {verdict}")
    print("  (15 m appears only in a diagnostic-cell reassignment + plot labels, not the link threshold)")
else:
    print("no V2 buildings parquet found")

match_distance present: 18,550 of 907,371 rows
  max=10.000 m | >10m=0 | >15m=0
  match_method counts:
match_method
NaN        888821
within      12304
nearest      6246

  OPERATIVE RADIUS = 10 m
  (15 m appears only in a diagnostic-cell reassignment + plot labels, not the link threshold)


## C7 - V3 point row count (raw vs filtered)

The 62,043 vs 63,243 discrepancy is raw-vs-filtered. Report both; cite the on-disk number for the unit you model.

In [4]:
tot_raw = tot_valid = 0
for t in TIERS:
    p = DS / "v3" / f"bda_points_t{t}.parquet"
    if p.exists():
        n = nrows(p)
        df = pd.read_parquet(p, columns=["damage_binary","city"])
        valid = int((df["damage_binary"]>=0).sum())
        tot_raw += (n or 0); tot_valid += valid
        print(f"  t{t}: raw={n:,}  damage_binary>=0={valid:,}  cities={df['city'].nunique()}")
print(f"\n  TOTAL raw={tot_raw:,}   valid(damage_binary>=0)={tot_valid:,}")
print("  docs cite 62,043 vs 63,243 -> the delta is the validity filter")

  t0: raw=39,997  damage_binary>=0=39,997  cities=4
  t1: raw=19,020  damage_binary>=0=19,020  cities=11
  t2: raw=4,226  damage_binary>=0=4,226  cities=6

  TOTAL raw=63,243   valid(damage_binary>=0)=63,243
  docs cite 62,043 vs 63,243 -> the delta is the validity filter


## C8 - y_true unit (building-level vs point-level)

V2 ML uses BUILDING-level `damage_binary`; V3-V7 ML uses POINT-level. Multiple UNOSAT points map to one building, so
point-positives > building-positives. State the unit with every count ("7,332 buildings" vs "10,567 points").

In [5]:
def pos_neg(sub, fname, has_bid):
    pos=neg=tot=0
    for t in TIERS:
        p = DS / sub / f"{fname}_t{t}.parquet"
        if p.exists():
            usecols = ["damage_binary"] + (["building_id"] if has_bid else [])
            df = pd.read_parquet(p, columns=usecols)
            pos += int((df["damage_binary"]==1).sum())
            neg += int((df["damage_binary"]==0).sum())
            tot += len(df)
    return pos, neg, tot

bp = pos_neg("V2","bda_buildings", False)
pp = pos_neg("v3","bda_points", True)
print(f"V2 buildings : positives={bp[0]:,}  negatives={bp[1]:,}  total={bp[2]:,}")
print(f"v3 points    : positives={pp[0]:,}  negatives={pp[1]:,}  total={pp[2]:,}")
if bp[0]: print(f"point/building positive ratio: {pp[0]/bp[0]:.2f}")
print("\nUNIT: V2 y_true = BUILDING-level; V3-V7 y_true = POINT-level. Always state the unit.")

V2 buildings : positives=7,332  negatives=591,263  total=907,371
v3 points    : positives=8,247  negatives=54,996  total=63,243
point/building positive ratio: 1.12

UNIT: V2 y_true = BUILDING-level; V3-V7 y_true = POINT-level. Always state the unit.


## Footprint source - Overture (primary) vs OSM (comparison)

This pipeline matches damage to **Overture** footprints and also computes the **OSM** match rate for comparison (NB01
01b). The two comparator theses use OSM only; OSM completeness in Ukrainian conflict cities is uneven, so
Overture-primary is a methodological strength. Footprint-completeness caveat: a destroyed building absent from the
footprint layer is dropped from the denominator, biasing per-city damage rates downward.

In [6]:
search_dirs = [DRIVE_ROOT / "data", RESULTS_ROOT]
hits = []
for d in search_dirs:
    if d.exists():
        for pat in ["*match_rate*.json","*overture*osm*.json","*match*stat*.json"]:
            hits += list(d.rglob(pat))
hits = sorted(set(hits))[:15]
print("Overture/OSM match-rate stat files (from NB01 01b):")
for h in hits:
    print("  ", h.relative_to(DRIVE_ROOT))
if not hits:
    print("  none found on disk - the comparison runs inside NB01 01b; re-run/export if you want the stats persisted")
print("\nNote: Overture primary; OSM secondary (comparators use OSM only). Completeness caveat = denominator bias.")

OSError: [Errno 12] Cannot allocate memory: '/content/drive_f/masterthesis/data/satellite/landuse_classification/Kramatorsk/crossbattle/20230126'

## OOF coverage inventory (BOTH roots)

OOF is **not all in one place**. Two roots:
1. `RESULTS_ROOT/<nb>/oof_predictions/` - nb07, nb08, nb08b, nb09a(+v3/v4/v5), nb09b, nb09c, nb09e, nb10b
2. `DRIVE_ROOT/data/outputs/<NB>/...` - NB08c_v2-v7, NB09a_v2, NB10a, **NB11_V2/oof**, NB11x_*

NB11 (Optuna) OOF is generated by the dedicated NB13 OOF notebook into `data/outputs/NB11_V2/oof`. N1 (mean-folds vs
pooled) must read BOTH roots.

In [7]:
OOF_ROOTS = []
for attr in ["RESULTS_ROOT","OUTPUT_ROOT","DATA_OUTPUTS"]:
    v = getattr(gs, attr, None)
    if v: OOF_ROOTS.append(Path(v))
OOF_ROOTS += [DRIVE_ROOT / "data" / "outputs", RESULTS_ROOT]
seen=set(); roots=[]
for r in OOF_ROOTS:
    if r.exists() and r not in seen:
        roots.append(r); seen.add(r)

inv = collections.Counter(); total = 0
for root in roots:
    for p in root.rglob("oof_*.parquet"):
        inv[(root.name, p.relative_to(root).parts[0])] += 1; total += 1
print(f"OOF parquet coverage across {len(roots)} roots ({total} files):")
last=None
for (rootname, grp), n in sorted(inv.items()):
    if rootname != last:
        print(f"  [{rootname}]"); last=rootname
    print(f"     {grp:28s} {n}")

print("\nexperiment groups with OOF (either root):", sorted({g for (_, g) in inv}))
print("\nFor N1: point the OOF loader at BOTH roots above.")
print("Reruns 9d/10d/11x: confirm OOF lands in one of these roots (11x -> data/outputs/NB11_V2/oof via the NB13 OOF notebook).")

OOF parquet coverage across 2 roots (1793 files):
  [outputs]
     NB08c_v2                     347
     NB08c_v3                     70
     NB08c_v4                     70
     NB08c_v5                     70
     NB08c_v6                     70
     NB08c_v7                     70
     NB09a_v2                     32
     NB10a_v1                     46
     NB11_V2                      47
     NB11x_V2                     3
     NB11x_V3                     3
  [results]
     nb07                         157
     nb07v3                       41
     nb08                         49
     nb08b                        77
     nb09a                        85
     nb09a_v3                     96
     nb09a_v4                     48
     nb09a_v5                     48
     nb09b_v2                     92
     nb09b_v3                     59
     nb09c_v2                     61
     nb09c_v3                     39
     nb09e_v2                     18
     nb10b_v3                     95



## Summary - resolved values

| ID | Item | Resolution |
|---|---|---|
| C2 | feature count | No single number: tier-dependent (block_stats V2 t0/t1/t2 = 745/183/897), differs V2 vs V3-V7; cite the SHARED-across-tiers feature set for the modelled unit; dietrich28 is a column subset |
| C6 | match radius | Operative = 10 m (provable from match_distance.max()); 15 m is a stale reassignment + plot label in a diagnostic cell only |
| C7 | V3 rows | raw vs filtered delta; cite on-disk valid count |
| C8 | y_true unit | building-level (V2) vs point-level (V3-V7); points-per-building > 1; state unit every time |
| - | footprint | Overture primary, OSM comparison in NB01; comparators use OSM only; completeness caveat = denominator bias |
| OOF | coverage | TWO roots: results/<nb>/oof_predictions AND data/outputs/<NB>/...; 11x OOF under data/outputs/NB11_V2/oof (made by the NB13 OOF notebook); N1 reads both |